In [ ]:
import pandas as pd
import numpy as np
import requests
from tqdm.auto import tqdm

# Profile Literature

Describe the literature set identified from previous notebook. Use citation measures identified from trusted sources to help further prune and select our relevant literature set for interaction curation task. 

## Load Data

Using the recent PMIDs set as the previous kernel crashed from lack of memory. This will mean we have to regrab the text down the line but we should be ok to do this. TODO: Figure out how to chunk the text grab task better to handle larger data sets.

In [ ]:
df = pd.read_excel('../../../data/2026-07-02-test03-oncology-genes-recent-pmids.xlsx')

## Define Citation Measurements

Load supplementary datasets and create functions for relevant citation metrics. The metrics identified are the SJR quantile measures, article-level H-index, reciprocal citation rate (RCR), and NIH Citation Percentile (via RCR score). These will be obtained through Scimago and iCite resources.

In [ ]:
tqdm.pandas()
session = requests.Session()


In [ ]:
# Load SCImago once
sjr_df = pd.read_csv(
    "../../../data/scimagojr2025.csv",
    sep=";",
    decimal=",",
)

sjr_df["journal_norm"] = (
    sjr_df["Title"]
    .str.lower()
    .str.strip()
)

# Keep relevant columns
sjr_df = sjr_df[
    [
        "journal_norm",
        "SJR",
        "H index",
        "SJR Best Quartile",
        "Country",
        "Coverage",
    ]
]

# Lookup functions
def grab_sjr(journal):

    if pd.isna(journal):
        return np.nan

    journal = journal.lower().strip()

    match = sjr_df[sjr_df["journal_norm"] == journal]

    if match.empty:
        return np.nan

    return match.iloc[0]["SJR"]


def grab_h_index(journal):

    if pd.isna(journal):
        return np.nan

    journal = journal.lower().strip()

    match = sjr_df[sjr_df["journal_norm"] == journal]

    if match.empty:
        return np.nan

    return match.iloc[0]["H index"]

In [ ]:
# NIH iCite
def grab_iCite(pmid):

    try:

        url = f"https://icite.od.nih.gov/api/pubs?pmids={pmid}"

        r = session.get(url, timeout=30)

        if r.status_code != 200:
            return {
                "citation_count": np.nan,
                "rcr": np.nan,
                "nih_percentile": np.nan,
            }

        papers = r.json().get("data", [])

        if len(papers) == 0:
            return {
                "citation_count": np.nan,
                "rcr": np.nan,
                "nih_percentile": np.nan,
            }

        p = papers[0]

        return {
            "citation_count": p.get("citation_count"),
            "rcr": p.get("relative_citation_ratio"),
            "nih_percentile": p.get("nih_percentile"),
        }

    except Exception:

        return {
            "citation_count": np.nan,
            "rcr": np.nan,
            "nih_percentile": np.nan,
        }

## Grab Metrics

Functions and supplementary datasets are in place, use them to grab the relevant statistics and measures.

In [ ]:
metrics = df["pmid"].progress_apply(grab_iCite).apply(pd.Series)
df = pd.concat([df, metrics], axis=1)

df["sjr"] = df["journal"].progress_apply(grab_sjr)

df["h_index"] = df["journal"].progress_apply(grab_h_index)

# Citation velocity?
CURRENT_YEAR = 2026
df["citations_per_year"] = (
    df["citation_count"] /
    (CURRENT_YEAR - df["publication_year"] + 1)
)

In [ ]:
df.head()

In [ ]:
# Checkpoint
df.to_csv('../../../data/2026-07-06-test03-pmid-statistics.csv')

## Numbers


A look at some statistics / distributions for the literature in question. Get an idea of what we are working with and ideas for how to break this down and make the fetch much more managable (i.e. not crash).

In [ ]:
df.head()

In [ ]:
import plotly.express as px
import plotly

px.histogram(
    df,
    x="rcr",
    nbins=40,
    title="Relative Citation Ratio"
)

In [ ]:
px.histogram(
    df,
    x="nih_percentile",
    nbins=40,
    title="NIH Percentile"
)

In [ ]:
px.histogram(
    df,
    x="sjr",
    nbins=40,
    title="Journal SJR"
)

In [ ]:
px.scatter(
    df,
    x="sjr",
    y="rcr",
    hover_data=["title", "journal"],
    title="Article Impact versus Journal Prestige",
)

In [ ]:
thresholds = [90, 95, 98, 99, 99.9]

print(f"Total papers: {len(df):,}\n")

for threshold in thresholds:
    n = (df["nih_percentile"] >= threshold).sum()
    pct = 100 - threshold
    print(f"Top {pct:g}% (NIH percentile ≥ {threshold}): {n:,} papers")